# NB02 — Predict metal mobility from MAG density

**Goal:** Test hypotheses H1, H2, H3 via spatial block CV.

- **H1:** M1 (metal-gene density features) beats B0 (mean predictor) on at least 3/5 folds.
- **H2:** M1 (metal features only) beats M2 (non-metal env features only) on at least 3/5 folds.
- **H3:** M3 (all features) beats M1 on at least 3/5 folds (do non-metal features add signal?).

**Models:**
- B0: mean predictor
- B1: SoilGrids pH + OC only
- B2: climate features only (MAT, MAP, temp_seasonality)
- B3: all non-metal features
- M1: metal mobility fractions (PF1_Cu/Zn/Pb/Cr/Ni)
- M2: same as B3
- M3: M1 + M2 combined

**Primary target:** `PF1_Cu` (copper mobility fraction). Secondary: `PF1_Zn`, `PF1_Pb`.

**Output:** `data/cv_results.csv`, `data/shap_mean_abs.csv`, `figures/nb02_cv_rmse.png`, `figures/nb02_shap_bar.png`.

In [1]:
print("NB02 executing — predicting CSU metal mobility from MAG genomic density.")

NB02 executing — predicting CSU metal mobility from MAG genomic density.


In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from modelling import (
    MAG_DENSITY_FEATURES, NON_METAL_FEATURES, ALL_FEATURES, CSU_TARGETS,
    make_spatial_folds, spatial_block_cv, summarise_cv, build_xgboost,
)
from evaluation import (
    compute_shap_values, plot_shap_bar, shap_metal_fraction,
    format_cv_table,
)

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, FIGW, ROW_H, PALETTE, grid_h
apply_style()

DATA_DIR = Path.cwd().parent / 'data'
FIG_DIR = Path.cwd().parent / 'figures'
FIG_DIR.mkdir(exist_ok=True)

In [3]:
df = pd.read_parquet(DATA_DIR / 'mag_feature_matrix.parquet')
# Primary target: Cu mobility fraction.  Iterate over CSU_TARGETS for H1/H2 multi-target tests.
TARGET = 'PF1_Cu'

# Detect any subcategory density columns produced by batch_compute_densities
subcat_density_cols = [c for c in df.columns if c.startswith('ko_per_mb_') and c != 'ko_per_mb_primary']
MAG_ALL_DENSITY = MAG_DENSITY_FEATURES + subcat_density_cols
print(f"MAG density features: {MAG_ALL_DENSITY}")

# Require target + all predictor cols + coordinates
required_cols = [TARGET, 'latitude', 'longitude'] + MAG_ALL_DENSITY + NON_METAL_FEATURES
df_clean = df.dropna(subset=[c for c in required_cols if c in df.columns]).copy()

print(f"Clean rows for modelling: {len(df_clean):,}")
print(df_clean[TARGET].describe())

MAG density features: ['ko_per_mb_primary', 'ko_per_mb_resistance', 'ko_per_mb_transport', 'ko_per_mb_sensing', 'ko_per_mb_metabolism', 'ko_per_mb_cofactor']
Clean rows for modelling: 13,182
count    13182.000000
mean         0.098335
std          0.045290
min          0.028412
25%          0.067091
50%          0.082482
75%          0.103106
max          0.395291
Name: PF1_Cu, dtype: float64


In [4]:
# M1: MAG genomic density features only (primary hypothesis feature set)
# M2: non-metal env baselines only (SoilGrids)
# M3: MAG density + non-metal env combined
FEATURE_SETS = {
    'B1': ['ph_h2o', 'organic_carbon_density'],
    'B2': NON_METAL_FEATURES,
    'M1': MAG_ALL_DENSITY,
    'M2': NON_METAL_FEATURES,
    'M3': MAG_ALL_DENSITY + NON_METAL_FEATURES,
}
# Filter to cols present in df_clean
FEATURE_SETS = {k: [c for c in v if c in df_clean.columns] for k, v in FEATURE_SETS.items()}

In [5]:
cv_results = spatial_block_cv(
    df=df_clean,
    feature_sets=FEATURE_SETS,
    target_col=TARGET,
)
cv_summary = summarise_cv(cv_results)

cv_results.to_csv(DATA_DIR / 'cv_results.csv', index=False)
print(format_cv_table(cv_summary, model_order=['B0', 'B1', 'B2', 'B3', 'M1', 'M2', 'M3']).to_string(index=False))

model RMSE (mean ± SD) R² (mean ± SD)  n_folds
   B0    0.050 ± 0.021 -0.171 ± 0.221      5.0
   B1    0.055 ± 0.019 -0.540 ± 0.500      5.0
   B2    0.044 ± 0.022 -0.030 ± 0.764      5.0
   B3        nan ± nan      nan ± nan      NaN
   M1    0.053 ± 0.020 -0.341 ± 0.175      5.0
   M2    0.044 ± 0.022 -0.030 ± 0.764      5.0
   M3    0.040 ± 0.019  0.162 ± 0.443      5.0


In [6]:
rmse = cv_summary.set_index('model')['mean_rmse']
print("H1 (M1 < B0):", "SUPPORTED" if rmse['M1'] < rmse['B0'] else "NOT SUPPORTED")
print("H2 (M1 < M2):", "SUPPORTED" if rmse['M1'] < rmse['M2'] else "NOT SUPPORTED")
print("H3 (M3 < M1):", "SUPPORTED" if rmse['M3'] < rmse['M1'] else "NOT SUPPORTED")

H1 (M1 < B0): NOT SUPPORTED
H2 (M1 < M2): NOT SUPPORTED
H3 (M3 < M1): SUPPORTED


In [7]:
# SHAP on M3 (full model: MAG density + SoilGrids) to measure MAG density contribution
m3_features = FEATURE_SETS['M3']
valid_rows = df_clean.dropna(subset=m3_features + [TARGET])
X_shap = valid_rows[m3_features]
y_shap = valid_rows[TARGET].values

m3_full = build_xgboost()
m3_full.fit(X_shap.values, y_shap)

shap_values, mean_shap = compute_shap_values(m3_full, X_shap)
mean_shap.to_csv(DATA_DIR / 'shap_mean_abs.csv')

metal_frac = shap_metal_fraction(mean_shap, MAG_ALL_DENSITY)
print(f"MAG density SHAP fraction in M3: {metal_frac:.1%}")

plot_shap_bar(
    mean_shap,
    title=f"M3 SHAP importance — target: {TARGET}",
    metal_features=MAG_ALL_DENSITY,
    output_path=str(FIG_DIR / 'nb02_shap_bar.pdf'),
)

MAG density SHAP fraction in M3: 6.4%


In [8]:
model_order = [m for m in ['B0', 'B1', 'B2', 'M1', 'M2', 'M3'] if m in cv_summary['model'].values]
plot_data = cv_summary.set_index('model').reindex(model_order)

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H))
x = range(len(model_order))
colours = [PALETTE[0] if m.startswith('B') else PALETTE[1] for m in model_order]
ax.bar(x, plot_data['mean_rmse'], yerr=plot_data['sd_rmse'],
       color=colours, capsize=4, alpha=0.85, edgecolor='k', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(model_order)
ax.set_xlabel('Model')
ax.set_ylabel('Mean RMSE (5-fold spatial CV)')
ax.set_title(f'CV RMSE by model — target: {TARGET}')
grid_h(ax)
save(fig, FIG_DIR / 'nb02_cv_rmse')